# Paper Benchmark Table

Load selected model checkpoints and benchmark data, then build a compact results table for paper drafts. The first pass supports ERPFN checkpoints; competitor model hooks are left explicit for later additions.

In [ ]:
from pathlib import Path
import gc
import json
import os
import time
import warnings

import numpy as np
import pandas as pd
import psutil
import torch
from IPython.display import display
from sklearn.metrics import (
    adjusted_rand_score,
    average_precision_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)

from model import NanoERPFNLinker, NanoERPFNModel
from tokenizer import Tokenizer
from train import load_benchmarks, load_benchmark_frame
from utils import get_default_device, set_randomness_seed

set_randomness_seed(0)
device = get_default_device()
process = psutil.Process()
print(f"device: {device}")

## Configuration

In [ ]:
MANIFEST_PATH = Path("benchmark_data/manifest.json")
OUTPUT_DIRECTORY = Path("results/paper_benchmark_tables")
OUTPUT_PATH = OUTPUT_DIRECTORY / "benchmark_table_warm_start_cross_source_eval.csv"

# Set True when SentenceTransformer weights are already cached and the runtime
# should not attempt Hugging Face network checks.
HF_HUB_OFFLINE = True
if HF_HUB_OFFLINE:
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"

CHECKPOINT_SPECS = [
    {
        "model_name": "ERPFN WAG id-backend warm-start best 1250 cross-source eval",
        "checkpoint_path": Path(
            "results/model_checkpoints/20260724_083700_seed1337_f18a68a4_best_step1250.pt"
        ),
    },
    {
        "model_name": "ERPFN WAG id-backend warm-start final 1500 cross-source eval",
        "checkpoint_path": Path(
            "results/model_checkpoints/20260724_083700_seed1337_f18a68a4_last_step1500.pt"
        ),
    },
    {
        "model_name": "ERPFN WAG 600 cross-source eval",
        "checkpoint_path": Path(
            "results/model_checkpoints/20260715_170253_seed1337_3d8910a4_best_step0600.pt"
        ),
    },
    {
        "model_name": "ERPFN WAG 1500 cross-source eval",
        "checkpoint_path": Path(
            "results/model_checkpoints/20260715_195803_seed1337_20005214_best_step1500.pt"
        ),
    },
    {
        "model_name": "ERPFN WAG hard-neg 1500 cross-source eval",
        "checkpoint_path": Path(
            "results/model_checkpoints/20260721_200947_seed1337_bda60282_best_step1500.pt"
        ),
    },
]

BENCHMARK_NAMES = [
    "nc_voters_shared",
    "nc_voters_changed",
    "fodors_zagats",
    "dblp_acm",
    "dblp_google_scholar",
    "amazon_google_price_numeric",
    "amazon_google_price_both",
    "walmart_amazon",
    "bpid_matching_balanced",
    "ice_id_people_200",
]

CROSS_SOURCE_BENCHMARKS = {
    "fodors_zagats",
    "dblp_acm",
    "dblp_google_scholar",
    "amazon_google_price_numeric",
    "amazon_google_price_both",
    "walmart_amazon",
}

# Placeholder for later competitor integrations. Each competitor should produce
# the same row schema as evaluate_checkpoint_on_benchmarks().
COMPETITOR_SPECS = []

## Helpers

In [ ]:
def current_rss_mb() -> float:
    return process.memory_info().rss / (1024 * 1024)


def current_device_memory_mb() -> float | None:
    torch_device = torch.device(device)
    if torch_device.type == "cuda" and torch.cuda.is_available():
        return torch.cuda.memory_allocated(torch_device) / (1024 * 1024)
    if torch_device.type == "mps" and hasattr(torch.mps, "current_allocated_memory"):
        return torch.mps.current_allocated_memory() / (1024 * 1024)
    return None


def load_erpfn_checkpoint(checkpoint_path: Path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    run_config = checkpoint["run_config"]
    tokenizer = Tokenizer(
        text_backend=run_config["text_backend"],
        identifier_backend=run_config.get("identifier_backend"),
        normalize_identifiers=run_config.get("normalize_identifiers", False),
        default_phone_region=run_config.get("default_phone_region", "US"),
    )
    model = NanoERPFNModel(
        embedding_size=run_config["embedding_size"],
        text_embedding_size=tokenizer.text_embedding_dim,
        identifier_embedding_size=tokenizer.identifier_embedding_dim,
        num_attention_heads=run_config["num_attention_heads"],
        mlp_hidden_size=run_config["mlp_hidden_size"],
        num_layers=run_config["num_layers"],
        max_categories=run_config["max_categories"],
        record_representation=run_config.get("record_representation", "mean_pool"),
        adjacency_decoder=run_config.get("adjacency_decoder", "pair_mlp"),
        entity_slot_count=run_config.get("entity_slot_count", 300),
        num_slot_attention_layers=run_config.get("num_slot_attention_layers", 1),
    ).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return checkpoint, run_config, model, tokenizer


def pair_targets(benchmark: dict) -> tuple[np.ndarray, np.ndarray, str, int]:
    entity_ids = benchmark["entity_ids"]
    true_adjacency = entity_ids[:, None] == entity_ids[None, :]
    pair_mask = np.triu(np.ones_like(true_adjacency, dtype=bool), k=1)
    pair_scope = "all_pairs"
    excluded_same_source_pairs = 0

    if benchmark.get("pair_scope") == "paired_rows":
        pair_mask = np.zeros_like(true_adjacency, dtype=bool)
        pair_indices = benchmark["paired_row_indices"]
        pair_mask[pair_indices[:, 0], pair_indices[:, 1]] = True
        return (
            benchmark["paired_row_targets"].astype(bool),
            pair_mask,
            "paired_rows",
            0,
        )

    if benchmark["name"] in CROSS_SOURCE_BENCHMARKS:
        source_values = benchmark.get("source_table_values")
        if source_values is not None:
            cross_source_mask = source_values[:, None] != source_values[None, :]
            excluded_same_source_pairs = int((pair_mask & ~cross_source_mask).sum())
            pair_mask &= cross_source_mask
            pair_scope = "cross_source"

    return true_adjacency[pair_mask], pair_mask, pair_scope, excluded_same_source_pairs


def load_benchmark_metadata(manifest_path: Path, names: list[str]) -> pd.DataFrame:
    with manifest_path.open() as file:
        manifest = json.load(file)
    rows = []
    for name in names:
        config = manifest[name]
        rows.append(
            {
                "benchmark": name,
                "short_description": config.get(
                    "short_description", config["description"]
                ),
                "text_profile": config.get("text_profile", pd.NA),
                "field_summary": "; ".join(
                    f"{field}:{field_type}"
                    for field, field_type in config["field_types"].items()
                ),
                "description": config["description"],
            }
        )
    return pd.DataFrame(rows)


def metric_row(model_name: str, checkpoint_info: dict, benchmark: dict, linker):
    gc.collect()
    if torch.cuda.is_available() and torch.device(device).type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    rss_before = current_rss_mb()
    device_memory_before = current_device_memory_mb()
    start = time.perf_counter()
    predicted_labels = linker.fit_predict(
        benchmark["records"], benchmark["field_types"]
    )
    eval_seconds = time.perf_counter() - start
    rss_after = current_rss_mb()
    device_memory_after = current_device_memory_mb()

    targets, pair_mask, pair_scope, excluded_same_source_pairs = pair_targets(benchmark)
    probabilities = linker.adjacency_proba_[pair_mask]
    predictions = probabilities >= linker.threshold

    row = {
        "model_name": model_name,
        "model_kind": "erpfn_checkpoint",
        "run_id": checkpoint_info.get("run_id"),
        "checkpoint_step": checkpoint_info.get("step"),
        "checkpoint_label": checkpoint_info.get("label"),
        "benchmark": benchmark["name"],
        "n_records": len(benchmark["entity_ids"]),
        "n_entities": len(np.unique(benchmark["entity_ids"])),
        "n_fields": benchmark["records"].shape[1],
        "n_pairs": int(targets.size),
        "pair_scope": pair_scope,
        "candidate_pairs": int(targets.size),
        "excluded_same_source_pairs": excluded_same_source_pairs,
        "prevalence": float(targets.mean()),
        "eval_seconds": eval_seconds,
        "rss_mb_before": rss_before,
        "rss_mb_after": rss_after,
        "rss_mb_delta": rss_after - rss_before,
        "device_memory_mb_before": device_memory_before,
        "device_memory_mb_after": device_memory_after,
        "device_memory_mb_delta": None
        if device_memory_before is None or device_memory_after is None
        else device_memory_after - device_memory_before,
    }

    if np.unique(targets).size < 2:
        row.update(
            {
                "roc_auc": np.nan,
                "pr_auc": np.nan,
                "pair_precision_at_0_5": np.nan,
                "pair_recall_at_0_5": np.nan,
                "pair_f1_at_0_5": np.nan,
                "oracle_pair_f1": np.nan,
                "oracle_pair_threshold": np.nan,
                "adjusted_rand": np.nan,
            }
        )
        return row

    precision, recall, thresholds = precision_recall_curve(targets, probabilities)
    threshold_f1 = (
        2
        * precision[:-1]
        * recall[:-1]
        / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    )
    best_threshold_index = int(np.argmax(threshold_f1))

    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="The number of unique classes is greater than 50%.*",
            category=UserWarning,
        )
        adjusted_rand = (
            np.nan
            if pair_scope == "paired_rows"
            else adjusted_rand_score(benchmark["entity_ids"], predicted_labels)
        )

    row.update(
        {
            "roc_auc": roc_auc_score(targets, probabilities),
            "pr_auc": average_precision_score(targets, probabilities),
            "pair_precision_at_0_5": precision_score(
                targets, predictions, zero_division=0
            ),
            "pair_recall_at_0_5": recall_score(targets, predictions, zero_division=0),
            "pair_f1_at_0_5": f1_score(targets, predictions, zero_division=0),
            "oracle_pair_f1": float(threshold_f1[best_threshold_index]),
            "oracle_pair_threshold": float(thresholds[best_threshold_index]),
            "adjusted_rand": adjusted_rand,
        }
    )
    return row


def evaluate_checkpoint_on_benchmarks(
    spec: dict, benchmarks: list[dict]
) -> pd.DataFrame:
    checkpoint, run_config, model, tokenizer = load_erpfn_checkpoint(
        spec["checkpoint_path"]
    )
    linker = NanoERPFNLinker(model, tokenizer, device=device)
    rows = []
    for benchmark in benchmarks:
        print(f"{spec['model_name']} :: {benchmark['name']}")
        rows.append(metric_row(spec["model_name"], checkpoint, benchmark, linker))
    frame = pd.DataFrame(rows)
    frame.insert(1, "checkpoint_path", str(spec["checkpoint_path"]))
    frame.insert(4, "text_backend", run_config["text_backend"])
    frame.insert(
        5, "record_representation", run_config.get("record_representation", "mean_pool")
    )
    frame.insert(
        6, "adjacency_decoder", run_config.get("adjacency_decoder", "pair_mlp")
    )
    frame.insert(7, "train_generator", run_config.get("train_generator", pd.NA))
    return frame


def evaluate_competitor_on_benchmarks(
    spec: dict, benchmarks: list[dict]
) -> pd.DataFrame:
    raise NotImplementedError(
        "Add competitor evaluators here once baselines are selected."
    )

## Load Benchmark Data

In [ ]:
benchmark_metadata = load_benchmark_metadata(MANIFEST_PATH, BENCHMARK_NAMES)
benchmarks = load_benchmarks(MANIFEST_PATH, names=BENCHMARK_NAMES)

with MANIFEST_PATH.open() as file:
    manifest_config = json.load(file)

extra_benchmarks = []
extra_metadata_rows = []
for benchmark in benchmarks:
    config = manifest_config[benchmark["name"]]
    metadata_columns = [*config["field_types"].keys()]
    ignored_columns = config.get("ignored_columns", [])
    if "source_table" in ignored_columns:
        metadata_columns.insert(0, "source_table")
    if benchmark["name"] == "bpid_matching_balanced":
        metadata_columns = ["pair_id", "source_table", "match_label", *config["field_types"].keys()]

    metadata_frame = load_benchmark_frame(MANIFEST_PATH, config, metadata_columns)
    if benchmark["name"] in CROSS_SOURCE_BENCHMARKS and "source_table" in metadata_frame:
        benchmark["source_table_values"] = metadata_frame["source_table"].to_numpy()
    else:
        benchmark["source_table_values"] = None

    if benchmark["name"] == "bpid_matching_balanced":
        paired_indices = []
        paired_targets = []
        for _, group in metadata_frame.groupby("pair_id", sort=False):
            if set(group["source_table"]) != {"profile1", "profile2"} or len(group) != 2:
                continue
            left_index = int(group.index[group["source_table"] == "profile1"][0])
            right_index = int(group.index[group["source_table"] == "profile2"][0])
            paired_indices.append((min(left_index, right_index), max(left_index, right_index)))
            paired_targets.append(str(group["match_label"].iloc[0]).lower() == "true")

        paired_benchmark = benchmark.copy()
        paired_benchmark["name"] = "bpid_matching_paired"
        paired_benchmark["description"] = (
            "BPID labeled person-profile matching pairs evaluated on the original "
            "profile1/profile2 candidate pairs."
        )
        paired_benchmark["pair_scope"] = "paired_rows"
        paired_benchmark["paired_row_indices"] = np.asarray(paired_indices, dtype=int)
        paired_benchmark["paired_row_targets"] = np.asarray(paired_targets, dtype=bool)
        paired_benchmark["source_table_values"] = None
        extra_benchmarks.append(paired_benchmark)
        extra_metadata_rows.append(
            {
                "benchmark": "bpid_matching_paired",
                "short_description": "BPID paired candidates",
                "text_profile": config.get("text_profile", pd.NA),
                "field_summary": "; ".join(
                    f"{field}:{field_type}"
                    for field, field_type in config["field_types"].items()
                ),
                "description": paired_benchmark["description"],
            }
        )

benchmarks.extend(extra_benchmarks)
if extra_metadata_rows:
    benchmark_metadata = pd.concat(
        [benchmark_metadata, pd.DataFrame(extra_metadata_rows)], ignore_index=True
    )

benchmark_overview = pd.DataFrame(
    {
        "benchmark": benchmark["name"],
        "n_records": len(benchmark["entity_ids"]),
        "n_entities": len(np.unique(benchmark["entity_ids"])),
        "n_fields": benchmark["records"].shape[1],
        "field_types": ", ".join(benchmark["field_types"]),
        "description": benchmark["description"],
    }
    for benchmark in benchmarks
)
benchmark_overview = benchmark_metadata.merge(
    benchmark_overview, on="benchmark", how="left"
)
display(benchmark_overview)


## Evaluate Models

In [ ]:
frames = []
for spec in CHECKPOINT_SPECS:
    frames.append(evaluate_checkpoint_on_benchmarks(spec, benchmarks))

for spec in COMPETITOR_SPECS:
    frames.append(evaluate_competitor_on_benchmarks(spec, benchmarks))

results = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
if not results.empty:
    results = benchmark_metadata.merge(results, on="benchmark", how="right")
display(results)

## Paper Table

In [ ]:
paper_columns = [
    "model_name",
    "benchmark",
    "short_description",
    "text_profile",
    "n_records",
    "n_entities",
    "n_fields",
    "field_summary",
    "n_pairs",
    "pair_scope",
    "candidate_pairs",
    "excluded_same_source_pairs",
    "prevalence",
    "pr_auc",
    "pair_precision_at_0_5",
    "pair_recall_at_0_5",
    "pair_f1_at_0_5",
    "oracle_pair_f1",
    "oracle_pair_threshold",
    "adjusted_rand",
    "eval_seconds",
    "rss_mb_after",
    "device_memory_mb_after",
]

paper_table = results[paper_columns].copy()
numeric_columns = paper_table.select_dtypes(include="number").columns
display(paper_table.style.format({column: "{:.4f}" for column in numeric_columns}))

OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
results.to_csv(OUTPUT_PATH, index=False)
print(f"saved detailed benchmark table: {OUTPUT_PATH}")